# Parda — visual PII check: was the signature INK hidden?

On the unseen, phone-photographed pages, signatures scored 44.4% under the box rule (≥95% of the true box
blacked out). On a tilted photo the true box of a wide, thin signature is much bigger than the signature,
so that rule can fail even when every stroke is hidden. This notebook measures the **ink actually hidden**.

**Settings:** Accelerator **None (CPU)** — no GPU hours · Internet **On**
**Inputs:** **Datasets tab →** `celeba-dataset` (jessicali9530) · **Secrets:** `GITHUB_TOKEN`, `HF_TOKEN`

Run all cells in order (~15 min). It regenerates exactly the same 300 unseen pages as the real-world test.


In [ ]:
# 1. Code + secrets
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GH = secrets.get_secret("GITHUB_TOKEN")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
REPO, CODE = "chetanjagan/Parda", "/tmp/Parda"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "-q", f"https://{GH}@github.com/{REPO}.git", CODE], check=True)
    subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "-q", f"https://{GH}@github.com/{REPO}.git", "main"], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)


In [ ]:
# 2. Fonts (to regenerate the same pages) + ultralytics
!bash scripts/setup_kaggle.sh
!pip install -q ultralytics


In [ ]:
# 3. Held-out faces + your YOLO model
from huggingface_hub import HfApi
from parda.vision.faces import FacePool, find_faces_dir
FACES = find_faces_dir("/kaggle/input")
YOLO_REPO = HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"] + "/parda-yolo-v1"
print("FACES =", FACES, "|", FacePool(FACES).sizes() if FACES else "none", "| model:", YOLO_REPO)
assert FACES, "Add Input -> Datasets tab -> 'celeba-dataset' (jessicali9530)"
W = "/kaggle/working"


In [ ]:
# 4. Tests (~1 min): every line ok
!python tests/test_ink.py


In [ ]:
# 5. The same 300 unseen, photographed pages as the real-world test (same seed + faces) (~1 min)
!python -m parda.synth.ood --n 300 --out {W}/ood --faces_dir {FACES} --workers 4 > /dev/null
import json
stats = json.load(open(f"{W}/ood/stats.json"))
print("visual items:", stats["visuals"])
assert stats["visuals"] == {"FACE": 75, "SIGNATURE": 225, "QR_CODE": 48, "STAMP": 128}, "pages differ from the real-world test"
print("identical to the real-world test pages")


In [ ]:
# 6. INK CHECK (~5-10 min on CPU): box rule vs ink actually hidden, per class
!python -m parda.vision.ink_check --data {W}/ood --yolo {YOLO_REPO} --out {W}/ink --title "Unseen layouts + phone photos: visual PII" 2>&1 | grep -v Warning | tail -15
import glob
from IPython.display import Image as IPImage, Markdown, display
display(Markdown(open(f"{W}/ink/ink_check.md").read()))
print("Signatures that FAIL the box rule but have their ink hidden (original | redacted):")
for f in sorted(glob.glob(f"{W}/ink/examples/*.png"))[:6]:
    display(IPImage(f))


In [ ]:
# 7. Collect results to commit to GitHub (download from the Output tab)
!mkdir -p {W}/ink_results && cp {W}/ink/ink_check.md {W}/ink/ink_check.json {W}/ink_results/ && cp {W}/ink/examples/*.png {W}/ink_results/ 2>/dev/null; ls {W}/ink_results
